<a href="https://colab.research.google.com/github/nairanjelina-del/BookNest/blob/main/PythonCode.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [20]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score
)

RANDOM_STATE = 42



In [21]:
df = pd.read_csv("credit_default_600_sample_with_ID.csv")

predictor_cols = ["LIMIT_BAL", "SEX", "EDUCATION", "AGE", "PAY_0", "PAY_2",
                   "BILL_AMT1", "BILL_AMT2", "PAY_AMT1", "PAY_AMT2"]
target_col = "DEFAULT"

X = df[predictor_cols]
y = df[target_col]

In [22]:
print("\nColumn names:")
print(df.columns)

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isnull().sum())

print("\nTarget distribution:")
print(df["DEFAULT"].value_counts())

print("\nTarget distribution in percentage:")
print(df["DEFAULT"].value_counts(normalize=True) * 100)

print("\nDescriptive statistics:")
print(df.describe())



Column names:
Index(['ID', 'LIMIT_BAL', 'SEX', 'EDUCATION', 'AGE', 'PAY_0', 'PAY_2',
       'BILL_AMT1', 'BILL_AMT2', 'PAY_AMT1', 'PAY_AMT2', 'DEFAULT'],
      dtype='object')

Data types:
ID           int64
LIMIT_BAL    int64
SEX          int64
EDUCATION    int64
AGE          int64
PAY_0        int64
PAY_2        int64
BILL_AMT1    int64
BILL_AMT2    int64
PAY_AMT1     int64
PAY_AMT2     int64
DEFAULT      int64
dtype: object

Missing values:
ID           0
LIMIT_BAL    0
SEX          0
EDUCATION    0
AGE          0
PAY_0        0
PAY_2        0
BILL_AMT1    0
BILL_AMT2    0
PAY_AMT1     0
PAY_AMT2     0
DEFAULT      0
dtype: int64

Target distribution:
DEFAULT
0    467
1    133
Name: count, dtype: int64

Target distribution in percentage:
DEFAULT
0    77.833333
1    22.166667
Name: proportion, dtype: float64

Descriptive statistics:
                 ID      LIMIT_BAL         SEX   EDUCATION         AGE  \
count    600.000000     600.000000  600.000000  600.000000  600.000000   
mean

In [23]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE
)

print("\nTraining data:", X_train.shape)
print("Testing data:", X_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts(normalize=True))

print("\nTesting target distribution:")
print(y_test.value_counts(normalize=True))


Training data: (480, 10)
Testing data: (120, 10)

Training target distribution:
DEFAULT
0    0.779167
1    0.220833
Name: proportion, dtype: float64

Testing target distribution:
DEFAULT
0    0.775
1    0.225
Name: proportion, dtype: float64


In [24]:
categorical_features = [
    "SEX",
    "EDUCATION"
]

numeric_features = [
    "LIMIT_BAL",
    "AGE",
    "PAY_0",
    "PAY_2",
    "BILL_AMT1",
    "BILL_AMT2",
    "PAY_AMT1",
    "PAY_AMT2"
]

In [28]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numeric_features
        ),

        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ]
)


In [29]:
log_reg_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),

        (
            "classifier",
            LogisticRegression(
                class_weight="balanced",
                random_state=RANDOM_STATE,
                max_iter=1000
            )
        )
    ]
)

In [30]:
log_reg_pipeline.fit(
    X_train,
    y_train
)

print("\nLogistic Regression trained successfully.")




Logistic Regression trained successfully.


In [31]:


y_pred_lr = log_reg_pipeline.predict(X_test)

y_proba_lr = log_reg_pipeline.predict_proba(X_test)[:, 1]




In [32]:

accuracy_lr = accuracy_score(
    y_test,
    y_pred_lr
)

precision_lr = precision_score(
    y_test,
    y_pred_lr,
    zero_division=0
)

recall_lr = recall_score(
    y_test,
    y_pred_lr,
    zero_division=0
)

f1_lr = f1_score(
    y_test,
    y_pred_lr,
    zero_division=0
)

print("\n==============================")
print("LOGISTIC REGRESSION RESULTS")
print("==============================")

print("Accuracy :", round(accuracy_lr, 4))
print("Precision:", round(precision_lr, 4))
print("Recall   :", round(recall_lr, 4))
print("F1 Score :", round(f1_lr, 4))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_lr))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred_lr,
    zero_division=0
))



LOGISTIC REGRESSION RESULTS
Accuracy : 0.6083
Precision: 0.2826
Recall   : 0.4815
F1 Score : 0.3562

Confusion Matrix:
[[60 33]
 [14 13]]

Classification Report:
              precision    recall  f1-score   support

           0       0.81      0.65      0.72        93
           1       0.28      0.48      0.36        27

    accuracy                           0.61       120
   macro avg       0.55      0.56      0.54       120
weighted avg       0.69      0.61      0.64       120



In [34]:
# MODEL 2: DECISION TREE
tree_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),

        (
            "classifier",
            DecisionTreeClassifier(
                class_weight="balanced",
                max_depth=4,
                min_samples_split=10,
                min_samples_leaf=5,
                random_state=RANDOM_STATE
            )
        )
    ]
)


In [35]:
tree_pipeline.fit(
    X_train,
    y_train
)

print("\nDecision Tree trained successfully.")





Decision Tree trained successfully.


In [37]:


y_pred_tree = tree_pipeline.predict(X_test)




In [39]:

# 13. EVALUATE DECISION TREE


accuracy_tree = accuracy_score(
    y_test,
    y_pred_tree
)

precision_tree = precision_score(
    y_test,
    y_pred_tree,
    zero_division=0
)

recall_tree = recall_score(
    y_test,
    y_pred_tree,
    zero_division=0
)

f1_tree = f1_score(
    y_test,
    y_pred_tree,
    zero_division=0
)

print("\n==============================")
print("DECISION TREE RESULTS")
print("==============================")

print("Accuracy :", round(accuracy_tree, 4))
print("Precision:", round(precision_tree, 4))
print("Recall   :", round(recall_tree, 4))
print("F1 Score :", round(f1_tree, 4))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_tree))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred_tree,
    zero_division=0
))




DECISION TREE RESULTS
Accuracy : 0.675
Precision: 0.3571
Recall   : 0.5556
F1 Score : 0.4348

Confusion Matrix:
[[66 27]
 [12 15]]

Classification Report:
              precision    recall  f1-score   support

           0       0.85      0.71      0.77        93
           1       0.36      0.56      0.43        27

    accuracy                           0.68       120
   macro avg       0.60      0.63      0.60       120
weighted avg       0.74      0.68      0.70       120



In [40]:
print("\n======================================")
print("FINAL MODEL COMPARISON")
print("======================================")

final_results = pd.DataFrame({
    "Model": [
        "Logistic Regression",

        "Decision Tree"
    ],

    "Accuracy": [
        accuracy_lr,

        accuracy_tree
    ],

    "Precision": [
        precision_lr,

        precision_tree
    ],

    "Recall": [
        recall_lr,


        recall_tree
    ],

    "F1 Score": [
        f1_lr,

        f1_tree
    ]
})

print(final_results.round(4))


FINAL MODEL COMPARISON
                 Model  Accuracy  Precision  Recall  F1 Score
0  Logistic Regression    0.6083     0.2826  0.4815    0.3562
1        Decision Tree    0.6750     0.3571  0.5556    0.4348
